### This will not include passive OSINT

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [2]:
df = pd.read_csv("../data/Passive_features.csv")# don't forget this only has 10K data
df.head()

,url,type,url_length,domain,path,domain_length,path_length,num_dots,num_hyphens,num_underscores,...,has_suspicious_word,num_subdomains,domain_entropy,domain_age_days,ip,asn_org,country,region,is_dga_like,suspicious_domain
0,zowiemarketing.net,phishing,18,zowiemarketing.net,zowiemarketing.net,0,18,1,0,0,...,0,0,0.00000,-1,NaN,NaN,NaN,NaN,0,0
1,http://www.vilagnomad.com/tables/quick-loans-n...,malware,64,vilagnomad.com,/tables/quick-loans-no-credit-check.php,18,39,3,4,0,...,0,1,3.46132,-1,NaN,NaN,NaN,NaN,0,0
2,commons.wikimedia.org/wiki/Category:Soldier%27...,benign,69,commons.wikimedia.org,commons.wikimedia.org/wiki/Category:Soldier%27...,0,69,2,0,3,...,0,1,0.00000,-1,103.102.166.224,AS14907 Wikimedia Foundation Inc.,SG,Singapore,0,0
3,baseballcardshopper.com/Topps_Bowman_Fleer_Don...,benign,98,baseballcardshopper.com,baseballcardshopper.com/Topps_Bowman_Fleer_Don...,0,77,2,0,5,...,0,0,0.00000,6496,50.63.8.97,"AS26496 GoDaddy.com, LLC",US,Arizona,0,0
4,en.wikipedia.org/wiki/Psyche_Industry_Records,benign,45,en.wikipedia.org,en.wikipedia.org/wiki/Psyche_Industry_Records,0,45,2,0,2,...,0,1,0.00000,-1,103.102.166.224,AS14907 Wikimedia Foundation Inc.,SG,Singapore,0,0


In [3]:
print(df['type'].value_counts())

type
benign        1445
defacement     360
phishing       137
malware         58
Name: count, dtype: int64


* phishing --> Fake website pretending to be real
* Benign --> Normal, safe website
* Defacement --> Website that got hacked and altered
* Malware --> Website that infects your device

### URL for phishing URL ->
    https://sahe.in/jir/journal_management/production/plagiarism_files/2paper_12.pdf

In [4]:
print(df.columns)
print(df['type'].value_counts())
print(df['type'].dtype)

Index(['url', 'type', 'url_length', 'domain', 'path', 'domain_length',
       'path_length', 'num_dots', 'num_hyphens', 'num_underscores',
       'num_slashes', 'num_digits', 'num_letters', 'num_special_chars',
       'has_ip', 'has_at_symbol', 'has_double_slash_redirect',
       'has_https_token_in_domain', 'has_suspicious_word', 'num_subdomains',
       'domain_entropy', 'domain_age_days', 'ip', 'asn_org', 'country',
       'region', 'is_dga_like', 'suspicious_domain'],
      dtype='str')
type
benign        1445
defacement     360
phishing       137
malware         58
Name: count, dtype: int64
str


### Train and test 

In [5]:

X = df.drop(columns=['url', 'domain', 'path', 'type', 'ip', 'asn_org', 'country', 'region'])
y = df['type']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Label mapping:", label_mapping)

#Training 
X_train, X_test, y_train, y_test = train_test_split(X,y_encoded,test_size=0.2,random_state=42,stratify=y_encoded)

Label mapping: {'benign': np.int64(0), 'defacement': np.int64(1), 'malware': np.int64(2), 'phishing': np.int64(3)}


### Logistic Regression

In [6]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=2000,class_weight='balanced',n_jobs=-1)
log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)

print("Logistic Regression Results")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(classification_report(y_test,y_pred_lr,target_names=le.classes_))

c:\Users\Hanna Hahn\OneDrive\Documents\Cyber-projects\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Logistic Regression Results
Accuracy: 0.8625
              precision    recall  f1-score   support

      benign       0.98      0.90      0.94       289
  defacement       0.85      0.81      0.83        72
     malware       0.38      0.67      0.48        12
    phishing       0.39      0.67      0.49        27

    accuracy                           0.86       400
   macro avg       0.65      0.76      0.69       400
weighted avg       0.90      0.86      0.88       400



c:\Users\Hanna Hahn\OneDrive\Documents\Cyber-projects\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Random Forest

In [7]:

'''
I will first used a randomforest model since it's the data is a int/floats (aka jsut numbers) and see which features
are the main factor to determine would also help alot
'''


rf = RandomForestClassifier(n_estimators=400,random_state=42,n_jobs=-1,class_weight="balanced",max_depth=None)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test,y_pred,target_names=le.classes_))

Accuracy: 0.9600

Classification Report:
              precision    recall  f1-score   support

      benign       0.98      0.99      0.99       289
  defacement       0.87      0.94      0.91        72
     malware       0.88      0.58      0.70        12
    phishing       1.00      0.81      0.90        27

    accuracy                           0.96       400
   macro avg       0.93      0.83      0.87       400
weighted avg       0.96      0.96      0.96       400



In [8]:

#* Confusion matrix -> measure how well a classification model is performing

cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm,index=le.classes_,columns=le.classes_)
print("Confusion Matrix:")
print(cm_df)

Confusion Matrix:
            benign  defacement  malware  phishing
benign         287           1        1         0
defacement       4          68        0         0
malware          1           4        7         0
phishing         0           5        0        22


In [9]:

#* feature importance -> this will be good for reporting 

feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values(by="importance", ascending=False)

print("Top 15 features:")
print(feature_importance.head(15))

Top 15 features:
                feature  importance
6           num_slashes    0.186850
16       domain_entropy    0.132620
1         domain_length    0.117360
0            url_length    0.096029
2           path_length    0.095487
9     num_special_chars    0.088245
8           num_letters    0.067674
17      domain_age_days    0.050995
7            num_digits    0.041524
3              num_dots    0.034390
4           num_hyphens    0.030844
15       num_subdomains    0.027676
5       num_underscores    0.015616
14  has_suspicious_word    0.012439
18          is_dga_like    0.002208


### Graidient Boost

In [10]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=300,learning_rate=0.1,max_depth=3,random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

print("Gradient Boosting Results")
print(f"Accuracy: {accuracy_score(y_test, y_pred_gb):.4f}")
print(classification_report(y_test,y_pred_gb,target_names=le.classes_))

Gradient Boosting Results
Accuracy: 0.9550
              precision    recall  f1-score   support

      benign       0.98      0.99      0.99       289
  defacement       0.89      0.93      0.91        72
     malware       0.73      0.67      0.70        12
    phishing       0.91      0.78      0.84        27

    accuracy                           0.95       400
   macro avg       0.88      0.84      0.86       400
weighted avg       0.95      0.95      0.95       400



### XGBoost

In [11]:
from xgboost import XGBClassifier

xgb = XGBClassifier(objective='multi:softmax',
    num_class=len(le.classes_),
    n_estimators=400,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print("XGBoost Results")
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print(classification_report(y_test,y_pred_xgb,target_names=le.classes_))

XGBoost Results
Accuracy: 0.9650
              precision    recall  f1-score   support

      benign       0.98      0.99      0.99       289
  defacement       0.88      0.94      0.91        72
     malware       1.00      0.75      0.86        12
    phishing       1.00      0.81      0.90        27

    accuracy                           0.96       400
   macro avg       0.97      0.88      0.91       400
weighted avg       0.97      0.96      0.96       400



### Compare the accuracy score form each model

In [12]:
results = pd.DataFrame({
    "Model": ["Logistic Regression","Random Forest", "Gradient Boosting", "XGBoost"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test,y_pred),
        accuracy_score(y_test, y_pred_gb),
        accuracy_score(y_test, y_pred_xgb)
    ]
})

print(results)

                 Model  Accuracy
0  Logistic Regression    0.8625
1        Random Forest    0.9600
2    Gradient Boosting    0.9550
3              XGBoost    0.9650


### Compare Macro F1 score
Treats all classes equally, regardless of their frequency. Calculates the average F1 score (balance of precision and recall) independently for each class and then takes the unweighted mean

In [13]:
from sklearn.metrics import f1_score

'''
i feel that deciding by macro F1 might be best option since we should take the matrics and compare them equally, rather than
expecting them to all have the same amount of data
'''

print("Macro F1 Scores")
print("Logistic:", f1_score(y_test, y_pred_lr, average='macro'))
print("Random Forest:", f1_score(y_test, y_pred, average='macro'))
print("Gradient Boosting:", f1_score(y_test, y_pred_gb, average='macro'))
print("XGBoost:", f1_score(y_test, y_pred_xgb, average='macro'))

Macro F1 Scores
Logistic: 0.6872022163748588
Random Forest: 0.8731444143922629
Gradient Boosting: 0.8583559240787769
XGBoost: 0.9139513814743978


### Picking between the two modles

In [14]:
print("=========================  XGBoost =========================")
print(classification_report(y_test,y_pred_xgb,target_names=le.classes_))

print("=========================  Random Forest ====================")
print(classification_report(y_test,y_pred,target_names=le.classes_))

=========================  XGBoost =========================
              precision    recall  f1-score   support

      benign       0.98      0.99      0.99       289
  defacement       0.88      0.94      0.91        72
     malware       1.00      0.75      0.86        12
    phishing       1.00      0.81      0.90        27

    accuracy                           0.96       400
   macro avg       0.97      0.88      0.91       400
weighted avg       0.97      0.96      0.96       400

=========================  Random Forest ====================
              precision    recall  f1-score   support

      benign       0.98      0.99      0.99       289
  defacement       0.87      0.94      0.91        72
     malware       0.88      0.58      0.70        12
    phishing       1.00      0.81      0.90        27

    accuracy                           0.96       400
   macro avg       0.93      0.83      0.87       400
weighted avg       0.96      0.96      0.96       400



### Conclusion 
- RF finds more phishing URLs (higher recall)
- XGBoost is more conservative (higher precision)
- We will piorities Recall over percision with the logic of security